# 01 — Explore RRC Statewide Production Data (Oil)

**Phase 1 — Data Discovery and Understanding**

This notebook is exploratory only. It answers:

- What does one row represent?
- How many records exist?
- What fields exist, and what are their data types?
- Are there missing values?
- What years are included?
- Are there duplicate records?
- Are there invalid values?

Source file: `data/raw/production/PDF100.ebc` — an EBCDIC-encoded,
hierarchical mainframe extract from the Texas Railroad Commission, downloaded
from https://www.rrc.texas.gov/resource-center/research/data-sets-available-for-download/.
Format is documented from https://www.rrc.texas.gov/media/0a4dqvag/pda001.pdf
and summarized in `../docs/data_rrc_production.md`; read that first for the full
segment layout.

Key facts recap:
- Fixed **102-byte** binary records, EBCDIC encoded (assumed code page 037,
  validated against decoded data — see `../docs/data_rrc_production.md`)
- No delimiters — records must be read in fixed strides
- 24 possible segment types multiplexed in one file, identified by a
  2-byte key at the start of each record
- This notebook focuses on the 4 segments needed to reconstruct lease
  identity and monthly oil volumes: **Root (01)**, **Reporting Cycle
  (02)**, **Production (03)**, **Previous Production Report (07)**
- The raw file is **1.1 GB / ~10.8M records**. This notebook **streams
  through the entire file in a single pass** (chunked reads, never holding
  the whole 1.1 GB in memory at once) rather than sampling, so counts and
  ranges below are exact, not estimates.


In [29]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path("..") / "src"))

from oil_pipeline.extract import RECORD_LENGTH, load_pdf100

RAW_DATA_PATH = Path("..") / "data" / "raw" / "production" / "PDF100.ebc"
RAW_DATA_PATH.resolve()


WindowsPath('C:/texas-oil-data-platform/data/raw/production/PDF100.ebc')

## File size & record count

Confirm the file divides evenly into 102-byte records (validates the fixed-length assumption from the format spec) before reading anything.

In [30]:
file_size = RAW_DATA_PATH.stat().st_size
total_records, remainder = divmod(file_size, RECORD_LENGTH)

print(f"File size:        {file_size:,} bytes")
print(f"Record length:    {RECORD_LENGTH} bytes")
print(f"Total records:    {total_records:,}")
print(f"Remainder bytes:  {remainder} (should be 0 for a clean fixed-length file)")


File size:        1,100,339,178 bytes
Record length:    102 bytes
Total records:    10,787,639
Remainder bytes:  0 (should be 0 for a clean fixed-length file)


## What does one row represent?

One physical 102-byte record is **one segment of one of 24 possible types**
(see `../docs/data_rrc_production.md`), not one production month. The file
reading and EBCDIC/COMP-3 decoding logic lives in `src/oil_pipeline/`
(`extract.py`, `utils.py`), not in this notebook — `load_pdf100` below
streams the entire file once, tallying every record by its 2-byte type
key and parsing the four core segments as it goes, to see the full mix
(exact counts, not a sample estimate).

## Scan the full file

`load_pdf100` (in `src/oil_pipeline/extract.py`) streams all ~10.8M
records in a single pass: tallies every record by type key, and parses
the four core segments (Root, Reporting Cycle, Production, Previous
Production Report) into DataFrames. This is the expensive cell in this
notebook (reads the full 1.1 GB file); everything after it just analyzes
the results in memory.

In [31]:
import logging

logging.basicConfig(level=logging.INFO, format="%(message)s")

results = load_pdf100(RAW_DATA_PATH)

key_counts = results["key_counts"]
df_root = results["root"]
df_cycle = results["cycle"]
df_prod = results["production"]
df_prev = results["prev_production"]

assert key_counts["count"].sum() == total_records, "scanned count should match the file-size-derived total"


...2,000,000 records scanned
...4,000,000 records scanned
...6,000,000 records scanned
...8,000,000 records scanned
...10,000,000 records scanned
Done: 10,787,639 records scanned


In [32]:
key_counts


,key,count,segment,pct
0,02,2377134,PDORPTCY (Reporting Cycle Segment),22.04
1,03,1625955,PDOPROD (Production Segment),15.07
2,06,1374751,PDOCSHDS (Casinghead Disposition),12.74
3,05,1098789,PDODSP (Disposition & Stock Adjustment),10.19
4,04,998528,PDORMVDS (Discrepancy Removal Segment),9.26
5,13,769435,PDOPRVAL (Previous Allowable),7.13
6,08,686335,PDOCMPMT (Commingle Permit),6.36
7,09,686335,PDOCMPRD (Commingle Production),6.36
8,10,600477,PDOCMODS (Commingle Oil Disposition),5.57
9,07,326839,PDOPRPV (Previous Production Report),3.03


In [33]:
for name, df in [("Root (01)", df_root), ("Reporting Cycle (02)", df_cycle),
                  ("Production (03)", df_prod), ("Previous Prod Report (07)", df_prev)]:
    print(f"{name}: {len(df):,} rows")


Root (01): 165,436 rows
Reporting Cycle (02): 2,377,134 rows
Production (03): 1,625,955 rows
Previous Prod Report (07): 326,839 rows


## What fields exist, and what are their data types?

Preview each parsed segment. All columns come out as Python `str`/`int`
from the parsers above; `.info()` shows the resulting pandas dtypes.

In [34]:
df_prod.head()


,corrected_report_flag,oil_production_bbl,casinghead_gas_mcf,casinghead_gas_lift_mcf,batch_number,item_number,posting_year,posting_month,posting_day,filed_by_edi_flag
0,N,12,0,0,883,0647,2023,06,00,Y
1,N,14,0,0,859,0282,2023,05,00,Y
2,N,27,0,0,856,1175,2023,04,00,Y
3,N,14,0,0,853,0268,2023,03,00,Y
4,N,18,0,0,877,1210,2023,02,00,Y


In [35]:
df_prod.info()


<class 'pandas.DataFrame'>
RangeIndex: 1625955 entries, 0 to 1625954
Data columns (total 10 columns):
 #   Column                   Non-Null Count    Dtype
---  ------                   --------------    -----
 0   corrected_report_flag    1625955 non-null  str  
 1   oil_production_bbl       1625955 non-null  int64
 2   casinghead_gas_mcf       1625955 non-null  int64
 3   casinghead_gas_lift_mcf  1625955 non-null  int64
 4   batch_number             1625955 non-null  str  
 5   item_number              1625955 non-null  str  
 6   posting_year             1625955 non-null  str  
 7   posting_month            1625955 non-null  str  
 8   posting_day              1625955 non-null  str  
 9   filed_by_edi_flag        1625955 non-null  str  
dtypes: int64(3), str(7)
memory usage: 124.1 MB


## Are there missing values?

This is a fixed-format binary extract — every field is present in every
record by construction (no free-form nulls). "Missing" here instead means
**zero/blank-filled filler**: check for that instead of NaN.

In [8]:
print("Null counts (should all be 0 — fixed binary layout has no free-form nulls):")
print(df_prod.isna().sum())
print()
print("Records with zero oil production AND zero casinghead gas (all-zero volume rows):")
zero_volume = ((df_prod["oil_production_bbl"] == 0) & (df_prod["casinghead_gas_mcf"] == 0)).sum()
print(f"{zero_volume:,} / {len(df_prod):,} ({zero_volume / len(df_prod):.1%})")


Null counts (should all be 0 — fixed binary layout has no free-form nulls):


corrected_report_flag      0
oil_production_bbl         0
casinghead_gas_mcf         0
casinghead_gas_lift_mcf    0
batch_number               0
item_number                0
posting_year               0
posting_month              0
posting_day                0
filed_by_edi_flag          0
dtype: int64

Records with zero oil production AND zero casinghead gas (all-zero volume rows):
380,622 / 1,625,955 (23.4%)


## What years are included?

Two independent date sources exist: the reporting cycle key (**YYMM** —
confirmed from decoded data, see `../docs/data_rrc_production.md`, despite the
source PDF stating MMYY) on the Reporting Cycle segment, and the posting
date (`YYYYMMDD`) on the Production segment. Since the full file has been
scanned, these are the dataset's actual min/max, not a sample estimate.

In [9]:
cycle_keys = df_cycle["rpt_cycle_key_yymm"].sort_values()
print("Reporting cycle key (YYMM) — full file:")
print(f"  distinct values: {cycle_keys.nunique():,}")
print(f"  earliest: {cycle_keys.iloc[0]}   latest: {cycle_keys.iloc[-1]}")
print()
print("Production posting years — full file:")
print(df_prod["posting_year"].value_counts().sort_index())


Reporting cycle key (YYMM) — full file:


  distinct values: 26
  earliest: 2105   latest: 2306

Production posting years — full file:


posting_year
2021    360090
2022    825876
2023    439989
Name: count, dtype: int64


## Are there duplicate records?

`PDROOT` should have one record per lease, so `(district_code, lease_nbr)`
should be unique across the whole file.

In [10]:
dup_leases = df_root.duplicated(subset=["district_code", "lease_nbr"]).sum()
print(f"Duplicate (district_code, lease_nbr) pairs in Root segment (full file): {dup_leases:,} / {len(df_root):,}")

exact_dup_root_rows = df_root.duplicated().sum()
print(f"Fully duplicate Root rows: {exact_dup_root_rows:,}")


Duplicate (district_code, lease_nbr) pairs in Root segment (full file): 0 / 165,436


Fully duplicate Root rows: 0


## Are there invalid values?

Sanity-check values against what the format spec says they should be:
- `oil_code` should always be `O`
- `district_code` should be one of the 14 known encoded district values
- `corrected_report_flag` / `filed_by_edi_flag` should be `N`/`Y` (or blank)
- `posting_month` should be `01`–`12`
- production volumes should not be negative

In [11]:
# All 14 possible stored values per docs/data_rrc_production.md; "12" is documented
# as unused and is confirmed absent from this file, but is a legitimate defined
# code, so it stays in the "valid" set rather than being flagged as an error.
VALID_DISTRICT_CODES = {f"{i:02d}" for i in range(1, 15)}

invalid_oil_code = (df_root["oil_code"] != "O").sum()
invalid_district = (~df_root["district_code"].isin(VALID_DISTRICT_CODES)).sum()
invalid_corrected_flag = (~df_prod["corrected_report_flag"].isin(["N", "Y"])).sum()
invalid_edi_flag = (~df_prod["filed_by_edi_flag"].isin(["N", "Y", " "])).sum()
invalid_posting_month = (~df_prod["posting_month"].astype(int).between(1, 12)).sum()
negative_oil_production = (df_prod["oil_production_bbl"] < 0).sum()
negative_casinghead_gas = (df_prod["casinghead_gas_mcf"] < 0).sum()

print(f"Root — oil_code != 'O':               {invalid_oil_code:,} / {len(df_root):,}")
print(f"Root — unrecognized district_code:    {invalid_district:,} / {len(df_root):,}")
print(f"Prod — corrected_report_flag not N/Y: {invalid_corrected_flag:,} / {len(df_prod):,}")
print(f"Prod — filed_by_edi_flag not N/Y/' ': {invalid_edi_flag:,} / {len(df_prod):,}")
print(f"Prod — posting_month outside 1-12:    {invalid_posting_month:,} / {len(df_prod):,}")
print(f"Prod — negative oil_production_bbl:   {negative_oil_production:,} / {len(df_prod):,}")
print(f"Prod — negative casinghead_gas_mcf:    {negative_casinghead_gas:,} / {len(df_prod):,}")


Root — oil_code != 'O':               0 / 165,436
Root — unrecognized district_code:    0 / 165,436
Prod — corrected_report_flag not N/Y: 0 / 1,625,955
Prod — filed_by_edi_flag not N/Y/' ': 0 / 1,625,955
Prod — posting_month outside 1-12:    0 / 1,625,955
Prod — negative oil_production_bbl:   0 / 1,625,955
Prod — negative casinghead_gas_mcf:    0 / 1,625,955


## Summary & next steps

Full-file scan of all 10,787,639 records:

- One physical row = one 102-byte segment record of 1 of 24 types, not one
  production month — reconstructing a lease-month requires joining Root
  (01) + Reporting Cycle (02) + Production (03) records by their position
  in the hierarchy.
- Segment breakdown: 165,436 Root (unique leases), 2,377,134 Reporting
  Cycle, 1,625,955 Production, 326,839 Previous Production Report — the
  remaining ~5.3M records span the other 20 (non-core) segment types.
- No null values are possible in this fixed binary layout; "missingness"
  instead shows up as zero-filled fields (23.4% of Production records have
  zero oil AND zero casinghead gas — likely shut-in/zero-report leases).
- This is a **rolling 26-month window** (reporting cycles `2105`–`2306`,
  May 2021 – June 2023), not a multi-decade historical archive — older
  cycles are evidently rolled off the database. This is a real finding,
  not a sampling artifact.
- `cp037` EBCDIC decoding is validated end-to-end: `PD-OIL-CODE` is `O` on
  100% of Root records, and all 13 distinct decoded district codes exactly
  match the 13 expected values from `../docs/data_rrc_production.md` (only the
  documented-unused code `12` is absent, as expected).
- Zero duplicate leases, zero invalid flags/months/codes, zero negative
  volumes found anywhere in the file.

The file-reading and decoding logic has moved out of this notebook into
`src/oil_pipeline/extract.py` (segment parsing, `load_pdf100`) and
`src/oil_pipeline/utils.py` (EBCDIC/COMP-3 primitives) — this notebook now
just calls `load_pdf100` and analyzes the result. Remaining Phase 2 work
(per `CLAUDE.md`): package `oil_pipeline` properly (or add it to
`sys.path` more robustly than the notebook hack above), add tests for the
decoding utilities, expand `load.py`/`transform.py`, and convert to
Parquet.